# S5 — 視覺化精華：課堂練習【解答版】

> **對應**：`S5_visualization_essentials.ipynb` 課堂練習段  
> **資料**：`orders_enriched.csv`（S3 產出）  
> **重點**：送分題 / 核心題 / 挑戰題（Electronics 迷你儀表板）完整解答

---

## 教學提示

- **Seaborn 的 `palette` 警告**：新版 seaborn 若只給 `palette=` 而不給 `hue=`，會噴 `FutureWarning`。正確做法是把分組欄位丟給 `hue`，若圖例重複再用 `legend=False` 隱藏。
- **`plt.tight_layout()`**：subplot 邊界擠在一起時必加，能自動調整間距避免標題／軸標籤重疊。
- **中文字型**：`plt.rcParams['axes.unicode_minus'] = False` 讓負號正常顯示；本節標題皆用英文，避免字型缺字問題。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')

df = pd.read_csv(
    '../datasets/ecommerce/orders_enriched.csv',
    parse_dates=['order_date'],
)
df['month'] = df['order_date'].dt.to_period('M').astype(str)
print('資料形狀:', df.shape)
df.head()

---
## 🟢 送分題

1. 各品類的**訂單數**長條圖
2. `amount` 的**直方圖**（`sns.histplot`）

In [ ]:
# 送分題 1：各品類的訂單數長條圖
cat_counts = df['category'].value_counts().reset_index()
cat_counts.columns = ['category', 'order_count']

plt.figure(figsize=(8, 4))
sns.barplot(
    data=cat_counts,
    x='category', y='order_count',
    hue='category', palette='viridis', legend=False,
)
plt.title('Order Count by Category', fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Order Count')

# 柱子上標數字
for i, v in enumerate(cat_counts['order_count']):
    plt.text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# 送分題 2：amount 的直方圖
plt.figure(figsize=(9, 4))
sns.histplot(data=df, x='amount', bins=30, kde=True, color='steelblue')
plt.title('Order Amount Distribution', fontweight='bold')
plt.xlabel('Amount (NT$)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

**解題重點**：
- 訂單數 = `value_counts()` 或 `groupby().size()`；不要跟 `sum(amount)` 搞混。
- `histplot` 加 `kde=True` 會疊一條密度曲線，能更直觀看出分布形狀（是否右偏）。

---
## 🟡 核心題

1. 比較 **North vs South** 兩地區的月度營收趨勢（兩條線的折線圖）
2. 不同 **vip_level** 的客單價箱型圖

In [ ]:
# 核心題 1：North vs South 月度營收比較折線圖
ns_df = df[df['region'].isin(['North', 'South'])]
monthly_ns = (
    ns_df.groupby(['month', 'region'])['amount']
    .sum()
    .reset_index()
)

plt.figure(figsize=(10, 4))
sns.lineplot(
    data=monthly_ns,
    x='month', y='amount',
    hue='region', marker='o', linewidth=2,
)
plt.title('Monthly Revenue: North vs South', fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Revenue (NT$)')
plt.xticks(rotation=45)
plt.legend(title='Region')
plt.tight_layout()
plt.show()

In [ ]:
# 核心題 2：不同 vip_level 的客單價箱型圖
plt.figure(figsize=(9, 5))
sns.boxplot(
    data=df,
    x='vip_level', y='amount',
    hue='vip_level', palette='Set3', legend=False,
)
plt.title('Order Amount Distribution by VIP Level', fontweight='bold')
plt.xlabel('VIP Level')
plt.ylabel('Amount (NT$)')
plt.tight_layout()
plt.show()

**解題重點**：
- 兩條線折線圖：最簡單是用 `hue='region'`，seaborn 會自動分色、建立圖例。
- 用 `isin(['North', 'South'])` 先過濾，避免其他地區也被畫進來。
- 箱型圖比較 vip 等級時，能一次看到中位數差異 + 是否有大筆離群訂單。

---
## 🔴 挑戰題 — Electronics 迷你儀表板

做一份**只屬於 Electronics 品類**的 2×2 儀表板：

| 位置 | 圖 |
|---|---|
| (0,0) | 月度趨勢折線圖 |
| (0,1) | 地區營收排名長條圖 |
| (1,0) | 商品 Top 5 長條圖 |
| (1,1) | 金額分布直方圖 |

In [ ]:
# 先過濾：只留 Electronics
elec = df[df['category'] == 'Electronics'].copy()
print('Electronics 訂單數:', len(elec))
print('Electronics 總營收:', f"NT$ {elec['amount'].sum():,.0f}")

In [ ]:
# 預先準備四張圖需要的資料
elec_monthly = elec.groupby('month')['amount'].sum().reset_index()
elec_region = (
    elec.groupby('region')['amount']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
elec_top5 = (
    elec.groupby('product_name')['amount']
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)

# 2x2 儀表板
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Electronics Category Dashboard', fontsize=16, fontweight='bold')

# (0,0) 月度趨勢
sns.lineplot(
    data=elec_monthly, x='month', y='amount',
    marker='o', linewidth=2, color='#1f77b4', ax=axes[0, 0],
)
axes[0, 0].set_title('Monthly Revenue Trend')
axes[0, 0].set_xlabel('Month')
axes[0, 0].set_ylabel('Revenue (NT$)')
axes[0, 0].tick_params(axis='x', rotation=45)

# (0,1) 地區排名
sns.barplot(
    data=elec_region, x='region', y='amount',
    hue='region', palette='viridis', legend=False, ax=axes[0, 1],
)
axes[0, 1].set_title('Revenue by Region')
axes[0, 1].set_xlabel('Region')
axes[0, 1].set_ylabel('Revenue (NT$)')
for i, v in enumerate(elec_region['amount']):
    axes[0, 1].text(i, v, f'{v:,.0f}', ha='center', va='bottom', fontsize=9)

# (1,0) 商品 Top 5
sns.barplot(
    data=elec_top5, y='product_name', x='amount',
    hue='product_name', palette='magma', legend=False, ax=axes[1, 0],
)
axes[1, 0].set_title('Top 5 Products')
axes[1, 0].set_xlabel('Revenue (NT$)')
axes[1, 0].set_ylabel('Product')

# (1,1) 金額分布
sns.histplot(
    data=elec, x='amount', bins=25, kde=True,
    color='#d62728', ax=axes[1, 1],
)
axes[1, 1].set_title('Amount Distribution')
axes[1, 1].set_xlabel('Amount (NT$)')
axes[1, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

**解題重點**：
1. **先 filter 再畫圖**：`df[df['category']=='Electronics']` 是所有圖的共同起點，避免每張圖都重寫條件。
2. **subplot 存取**：`axes[row, col]` 拿到指定子圖；sns 函式用 `ax=` 指定繪製位置。
3. **Top N 排行**：`groupby().sum().sort_values(ascending=False).head(5)`，畫橫條圖（`x=amount, y=product_name`）商品名字比較好讀。
4. **避免 FutureWarning**：seaborn 新版要求 `palette` 搭配 `hue`；圖例重複就 `legend=False`。
5. **`plt.tight_layout()`** 在 `plt.show()` 前呼叫，自動把 suptitle、軸標籤排好。

🎉 這份 Electronics 儀表板就是你可以直接放進履歷 side project 的成果。

---
## 延伸：把儀表板存成 PNG

面試作品集或週報時，加一行 `plt.savefig()` 就能把圖輸出成高解析檔案。

In [ ]:
# 範例：存成 150 dpi 的 PNG（取消註解即可執行）
# fig.savefig('electronics_dashboard.png', dpi=150, bbox_inches='tight')
# print('已輸出 electronics_dashboard.png')